# Lab 7. Estimation, Model Identification, and Diagnostics

**MATH 7339: Machine Learning and Statistical Learning Theory 2**  
**Modern Time Series Analysis and Sequence Data**

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-07-estimation-identification-diagnostics-lab.ipynb)

## Purpose of this lab

In Chapters 1--6, we learned how time series differ from ordinary independent data, how stationarity and autocorrelation are defined, how AR/MA/ARMA models are built, and how forecasts can be interpreted as prediction from past information.

This lab focuses on the next practical question:

> Given one observed time series, how do we identify, estimate, diagnose, and compare candidate time-series models?

This is an independent-study lab. Each programming section begins with background and interpretation guidance before code.

## Learning goals

After completing this lab, you should be able to:

1. simulate AR, MA, and ARMA processes;
2. use ACF and PACF plots to propose candidate models;
3. estimate AR coefficients using the Yule--Walker equations;
4. estimate ARMA models using maximum likelihood through `statsmodels`;
5. compare candidate models using AIC and BIC;
6. diagnose residuals using plots, residual ACF, and the Ljung--Box test;
7. evaluate forecasts on a chronological train/test split;
8. explain why model selection must be guided by both statistics and diagnostics.

Throughout the lab, remember that a good model should not only fit the observed series. Its residuals should behave approximately like white noise.

## 0. Background: what does model identification mean?

A common workflow for ARMA-type modeling is:

1. **Visualize the series.** Look for trend, changing variance, outliers, and seasonality.
2. **Check stationarity.** ARMA models are designed for stationary series.
3. **Inspect ACF and PACF.** These help suggest AR, MA, or mixed ARMA structure.
4. **Fit several candidate models.** Do not trust only one model.
5. **Check residuals.** Residuals should look like white noise.
6. **Compare forecasts.** A model that looks good in-sample may forecast poorly.

For a stationary zero-mean ARMA model,

$$
X_t - \phi_1 X_{t-1} - \cdots - \phi_p X_{t-p}
= W_t + \theta_1 W_{t-1} + \cdots + \theta_q W_{t-q},
$$

where $W_t$ is white noise. The parameters are usually estimated from data, so we need diagnostics to check whether the estimated model is adequate.

### ACF/PACF rules of thumb

These are not absolute rules, but they are useful starting points.

| True model type | ACF pattern | PACF pattern |
|---|---|---|
| AR($p$) | decays gradually | cuts off after lag $p$ |
| MA($q$) | cuts off after lag $q$ | decays gradually |
| ARMA($p,q$) | decays gradually | decays gradually |
| white noise | nearly zero for lags $>0$ | nearly zero for lags $>0$ |

The phrase "cuts off" means the theoretical value is exactly zero after a certain lag. In finite samples, we only expect the sample values to be small, not exactly zero.

## 1. Setup

The code below imports the libraries used in this lab. We use only standard scientific Python tools plus `statsmodels` for time-series estimation and diagnostics.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(7339)
print("Libraries imported successfully.")

## 2. Simulate several stationary time-series models

Before fitting models to real data, it is important to know what the models look like when we already know the truth.

We will simulate:

1. an AR(2) process;
2. an MA(1) process;
3. an ARMA(1,1) process.

We include a burn-in period. The burn-in allows the recursion to move away from the arbitrary initial value $0$ before we keep the final sample.

In [ ]:
def simulate_ar2(phi1=0.6, phi2=-0.25, n=500, burnin=300, sigma=1.0, seed=1):
    local_rng = np.random.default_rng(seed)
    total = n + burnin
    w = local_rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(2, total):
        x[t] = phi1 * x[t-1] + phi2 * x[t-2] + w[t]
    return x[burnin:]


def simulate_ma1(theta=0.7, n=500, burnin=300, sigma=1.0, seed=2):
    local_rng = np.random.default_rng(seed)
    total = n + burnin
    w = local_rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = w[t] + theta * w[t-1]
    return x[burnin:]


def simulate_arma11(phi=0.55, theta=0.65, n=500, burnin=300, sigma=1.0, seed=3):
    local_rng = np.random.default_rng(seed)
    total = n + burnin
    w = local_rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(1, total):
        x[t] = phi * x[t-1] + w[t] + theta * w[t-1]
    return x[burnin:]

n = 500
x_ar2 = simulate_ar2(n=n)
x_ma1 = simulate_ma1(n=n)
x_arma11 = simulate_arma11(n=n)

data = pd.DataFrame({
    "AR(2)": x_ar2,
    "MA(1)": x_ma1,
    "ARMA(1,1)": x_arma11
})

data.head()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, col in zip(axes, data.columns):
    ax.plot(data[col], linewidth=1)
    ax.set_title(col)
    ax.set_ylabel("value")

axes[-1].set_xlabel("time")
plt.tight_layout()
plt.show()

### Checkpoint 1

Look at the three plots above.

1. Do any of the series have an obvious trend?
2. Do they seem to fluctuate around a roughly constant mean?
3. Does the variability appear roughly stable over time?

These are visual signs of stationarity, but they are not a proof. We now use ACF and PACF to study dependence more directly.

## 3. ACF and PACF from the data

The **sample autocorrelation function** estimates the correlation between $X_t$ and $X_{t-h}$ at lag $h$.

For a zero-mean series, a natural sample autocovariance is

$$
\widehat{\gamma}(h)
= \frac{1}{n}\sum_{t=h+1}^{n} (x_t - \bar{x})(x_{t-h}-\bar{x}).
$$

The sample ACF is

$$
\widehat{\rho}(h)=\frac{\widehat{\gamma}(h)}{\widehat{\gamma}(0)}.
$$

The PACF measures the additional linear association at lag $h$ after removing the information already explained by lags $1,2,\ldots,h-1$.

In [ ]:
def sample_acf_from_scratch(x, max_lag):
    x = np.asarray(x)
    x_centered = x - x.mean()
    n = len(x)
    gamma0 = np.sum(x_centered * x_centered) / n
    values = []
    for h in range(max_lag + 1):
        gamma_h = np.sum(x_centered[h:] * x_centered[:n-h]) / n
        values.append(gamma_h / gamma0)
    return np.array(values)

max_lag = 25
acf_ar2_scratch = sample_acf_from_scratch(x_ar2, max_lag)
acf_ar2_statsmodels = acf(x_ar2, nlags=max_lag, fft=False)

pd.DataFrame({
    "lag": np.arange(max_lag + 1),
    "ACF from scratch": acf_ar2_scratch,
    "ACF from statsmodels": acf_ar2_statsmodels
}).head(10)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(12, 9))

for i, col in enumerate(data.columns):
    plot_acf(data[col], lags=30, ax=axes[i, 0], zero=True)
    axes[i, 0].set_title(f"ACF: {col}")
    plot_pacf(data[col], lags=30, ax=axes[i, 1], zero=True, method="ywm")
    axes[i, 1].set_title(f"PACF: {col}")

plt.tight_layout()
plt.show()

### Checkpoint 2

Use the ACF/PACF plots to answer these questions.

1. Which series looks most like an AR model?
2. Which series has an ACF that cuts off most quickly?
3. Which series looks like a mixed ARMA model?

Do not expect perfect textbook behavior. Sample ACF/PACF plots are random estimates.

## 4. Estimating an AR model by Yule--Walker equations

For a stationary AR($p$) model,

$$
X_t = \phi_1 X_{t-1} + \cdots + \phi_p X_{t-p} + W_t,
$$

multiplying both sides by $X_{t-h}$ and taking expectations gives the Yule--Walker equations. For AR(2), the equations are

$$
\begin{bmatrix}
\gamma(0) & \gamma(1) \\
\gamma(1) & \gamma(0)
\end{bmatrix}
\begin{bmatrix}
\phi_1 \\
\phi_2
\end{bmatrix}
=
\begin{bmatrix}
\gamma(1) \\
\gamma(2)
\end{bmatrix}.
$$

In practice, we replace population autocovariances by sample autocovariances.

In [ ]:
def sample_autocovariance(x, max_lag):
    x = np.asarray(x)
    x_centered = x - x.mean()
    n = len(x)
    values = []
    for h in range(max_lag + 1):
        values.append(np.sum(x_centered[h:] * x_centered[:n-h]) / n)
    return np.array(values)


def yule_walker_ar(x, p):
    gamma = sample_autocovariance(x, p)
    Gamma = np.empty((p, p))
    for i in range(p):
        for j in range(p):
            Gamma[i, j] = gamma[abs(i - j)]
    rhs = gamma[1:p+1]
    phi_hat = np.linalg.solve(Gamma, rhs)
    sigma2_hat = gamma[0] - np.dot(phi_hat, rhs)
    return phi_hat, sigma2_hat

phi_hat, sigma2_hat = yule_walker_ar(x_ar2, p=2)

print("Yule--Walker estimate for AR(2):")
print("phi_1 estimate:", round(phi_hat[0], 4))
print("phi_2 estimate:", round(phi_hat[1], 4))
print("innovation variance estimate:", round(sigma2_hat, 4))
print("true phi_1 = 0.60, true phi_2 = -0.25")

### Interpretation

The estimates will not equal the true parameters exactly because we only observe one finite sample. However, they should be reasonably close when the sample size is large and the model assumptions are correct.

Yule--Walker is especially natural for AR models. Mixed ARMA models usually require more complicated estimation, commonly maximum likelihood.

## 5. Fitting ARMA models with maximum likelihood

In Python, `statsmodels.tsa.arima.model.ARIMA` can fit ARMA models by using `order=(p,0,q)`.

The middle value is the differencing order $d$. Since the simulated examples in this lab are stationary, we use $d=0$.

For example:

```python
ARIMA(x, order=(1, 0, 1), trend="n")
```

fits an ARMA(1,1) model with no constant term.

In [ ]:
# Fit the true ARMA(1,1) model to the ARMA(1,1) data.
model_arma11 = ARIMA(x_arma11, order=(1, 0, 1), trend="n")
fit_arma11 = model_arma11.fit()

print(fit_arma11.summary())

### Checkpoint 3

In the output above, find:

1. the estimated AR coefficient;
2. the estimated MA coefficient;
3. the estimated innovation variance;
4. AIC and BIC.

Compare the estimates with the simulation values $\phi=0.55$ and $\theta=0.65$.

## 6. Grid search over candidate ARMA models

In real applications, we usually do not know $p$ and $q$. A simple strategy is to fit a small grid of candidate models and compare AIC and BIC.

For Gaussian likelihood models,

$$
\mathrm{AIC} = -2\ell + 2k,
$$

and

$$
\mathrm{BIC} = -2\ell + k\log(n),
$$

where $\ell$ is the maximized log-likelihood and $k$ is the number of estimated parameters. Lower values are better.

AIC often favors more flexible models. BIC penalizes complexity more strongly.

In [ ]:
def fit_arma_grid(x, p_max=3, q_max=3):
    records = []
    for p in range(p_max + 1):
        for q in range(q_max + 1):
            if p == 0 and q == 0:
                # White-noise model is allowed, but ARIMA can fit it too.
                pass
            try:
                fit = ARIMA(x, order=(p, 0, q), trend="n").fit()
                records.append({
                    "p": p,
                    "q": q,
                    "order": f"ARMA({p},{q})",
                    "AIC": fit.aic,
                    "BIC": fit.bic,
                    "loglik": fit.llf
                })
            except Exception as e:
                records.append({
                    "p": p,
                    "q": q,
                    "order": f"ARMA({p},{q})",
                    "AIC": np.nan,
                    "BIC": np.nan,
                    "loglik": np.nan
                })
    return pd.DataFrame(records).sort_values("AIC").reset_index(drop=True)

arma_grid = fit_arma_grid(x_arma11, p_max=3, q_max=3)
arma_grid

In [ ]:
print("Best model by AIC:")
print(arma_grid.loc[arma_grid["AIC"].idxmin(), ["order", "AIC", "BIC"]])

print("\nBest model by BIC:")
print(arma_grid.loc[arma_grid["BIC"].idxmin(), ["order", "AIC", "BIC"]])

### Checkpoint 4

The true model is ARMA(1,1). Did AIC and BIC select it?

If not, that does not automatically mean the method failed. In finite samples, nearby ARMA models can have very similar likelihoods. Model selection should combine:

- ACF/PACF evidence;
- AIC/BIC;
- residual diagnostics;
- forecast performance;
- interpretability and simplicity.

## 7. Residual diagnostics

After fitting a model, the residuals should behave approximately like white noise. We check this using:

1. a residual time plot;
2. a histogram;
3. residual ACF;
4. the Ljung--Box test.

The Ljung--Box test considers several residual autocorrelations together. A small p-value suggests that residual autocorrelation remains, so the model may be inadequate.

In [ ]:
resid = fit_arma11.resid

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(resid, linewidth=1)
axes[0].axhline(0, linestyle="--", linewidth=1)
axes[0].set_title("Residual time plot")
axes[0].set_xlabel("time")

axes[1].hist(resid, bins=30, edgecolor="black")
axes[1].set_title("Residual histogram")

plot_acf(resid, lags=30, ax=axes[2], zero=True)
axes[2].set_title("Residual ACF")

plt.tight_layout()
plt.show()

In [ ]:
ljung_box = acorr_ljungbox(resid, lags=[5, 10, 15, 20], return_df=True)
ljung_box

### Interpretation guide for the Ljung--Box table

- The null hypothesis is that the residuals have no autocorrelation up to the chosen lag.
- A large p-value means the residuals are consistent with white noise.
- A small p-value means the model has likely left time dependence unexplained.

Do not use the Ljung--Box test mechanically. Always combine it with residual plots and subject-matter judgment.

## 8. What happens when the model is wrong?

Now we intentionally fit a misspecified model to the ARMA(1,1) data. We fit an AR(1) model and compare its residual diagnostics with the ARMA(1,1) fit.

In [ ]:
fit_wrong_ar1 = ARIMA(x_arma11, order=(1, 0, 0), trend="n").fit()
wrong_resid = fit_wrong_ar1.resid

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(resid, lags=30, ax=axes[0], zero=True)
axes[0].set_title("Residual ACF: ARMA(1,1) fit")
plot_acf(wrong_resid, lags=30, ax=axes[1], zero=True)
axes[1].set_title("Residual ACF: wrong AR(1) fit")
plt.tight_layout()
plt.show()

comparison = pd.DataFrame({
    "model": ["ARMA(1,1)", "wrong AR(1)"],
    "AIC": [fit_arma11.aic, fit_wrong_ar1.aic],
    "BIC": [fit_arma11.bic, fit_wrong_ar1.bic],
    "Ljung-Box p-value at lag 20": [
        acorr_ljungbox(resid, lags=[20], return_df=True)["lb_pvalue"].iloc[0],
        acorr_ljungbox(wrong_resid, lags=[20], return_df=True)["lb_pvalue"].iloc[0]
    ]
})

comparison

### Checkpoint 5

Compare the two residual ACF plots and the table.

1. Which model leaves less autocorrelation in the residuals?
2. Which model has lower AIC and BIC?
3. Do the statistical diagnostics agree with the visual diagnostics?

## 9. Forecast evaluation with a chronological train/test split

Time series should not be randomly shuffled for forecasting evaluation. We train on the past and test on the future.

The following experiment:

1. uses the first 400 observations for training;
2. uses the last 100 observations for testing;
3. fits several candidate models on the training set;
4. forecasts the next 100 observations;
5. compares forecast errors.

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.asarray(y_true) - np.asarray(y_pred)) ** 2))

train_size = 400
train = x_arma11[:train_size]
test = x_arma11[train_size:]

candidate_orders = [(0, 0, 1), (1, 0, 0), (1, 0, 1), (2, 0, 0), (2, 0, 1)]
forecast_records = []
forecasts = {}

for order in candidate_orders:
    try:
        fit = ARIMA(train, order=order, trend="n").fit()
        forecast = fit.forecast(steps=len(test))
        name = f"ARMA({order[0]},{order[2]})"
        forecasts[name] = forecast
        forecast_records.append({
            "model": name,
            "AIC on train": fit.aic,
            "BIC on train": fit.bic,
            "MAE on test": mae(test, forecast),
            "RMSE on test": rmse(test, forecast)
        })
    except Exception as e:
        forecast_records.append({
            "model": f"ARMA({order[0]},{order[2]})",
            "AIC on train": np.nan,
            "BIC on train": np.nan,
            "MAE on test": np.nan,
            "RMSE on test": np.nan
        })

forecast_table = pd.DataFrame(forecast_records).sort_values("RMSE on test")
forecast_table

In [ ]:
plt.figure(figsize=(11, 5))
plt.plot(np.arange(len(x_arma11)), x_arma11, label="observed", linewidth=1)
plt.axvline(train_size, linestyle="--", label="train/test split")

for name, forecast in forecasts.items():
    if name in forecast_table["model"].head(3).values:
        plt.plot(np.arange(train_size, len(x_arma11)), forecast, label=f"forecast: {name}")

plt.title("Forecast comparison on holdout period")
plt.xlabel("time")
plt.ylabel("value")
plt.legend()
plt.show()

### Important forecasting lesson

For a stationary ARMA process, long-horizon forecasts often shrink toward the unconditional mean. If the model has mean zero and no deterministic trend, very long-run forecasts may look nearly flat.

This is not a bug. It is a property of stable stationary models.

## 10. Short exercise: diagnose a mystery series

Now we simulate a mystery stationary process. Your task is to inspect the plots, fit candidate models, and justify your conclusion.

In [ ]:
# Mystery series for student practice.
mystery = simulate_ar2(phi1=0.45, phi2=0.30, n=500, seed=2026)

plt.figure(figsize=(10, 4))
plt.plot(mystery, linewidth=1)
plt.title("Mystery series")
plt.xlabel("time")
plt.ylabel("value")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(mystery, lags=30, ax=axes[0], zero=True)
plot_pacf(mystery, lags=30, ax=axes[1], zero=True, method="ywm")
axes[0].set_title("Mystery ACF")
axes[1].set_title("Mystery PACF")
plt.tight_layout()
plt.show()

In [ ]:
mystery_grid = fit_arma_grid(mystery, p_max=3, q_max=3)
mystery_grid.head(8)

In [ ]:
# Choose one model after inspecting the ACF/PACF and the information criteria.
# Try changing this order and re-running the diagnostics.
chosen_order = (2, 0, 0)
fit_mystery = ARIMA(mystery, order=chosen_order, trend="n").fit()
resid_mystery = fit_mystery.resid

print(fit_mystery.summary())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(resid_mystery, linewidth=1)
axes[0].axhline(0, linestyle="--", linewidth=1)
axes[0].set_title("Mystery residuals")
plot_acf(resid_mystery, lags=30, ax=axes[1], zero=True)
axes[1].set_title("Mystery residual ACF")
plt.tight_layout()
plt.show()

acorr_ljungbox(resid_mystery, lags=[10, 20], return_df=True)

## 11. Student exercises

### Exercise 1: ACF/PACF identification

For each of the three original simulated series, write down one or two candidate models based only on the ACF and PACF plots.

### Exercise 2: Yule--Walker estimation

Use `yule_walker_ar` to fit AR(1), AR(2), and AR(3) models to the AR(2) data. Compare the coefficient estimates and residual diagnostics.

### Exercise 3: AIC versus BIC

Run the ARMA grid search for the MA(1) data and the AR(2) data. Does AIC or BIC choose the true model? Explain.

### Exercise 4: Residual diagnostics

Choose an intentionally wrong model for the MA(1) series. Fit it and make the residual ACF plot. What dependence remains?

### Exercise 5: Forecast comparison

Repeat the train/test forecasting experiment for the AR(2) data. Which candidate model forecasts best?

## 12. AI-assisted study prompts

You may use an AI assistant to help you study, but you must verify the output using your own code and reasoning.

### Prompt 1: ACF/PACF interpretation

> I have a stationary time series. Its sample ACF decays gradually and its sample PACF has large values at lags 1 and 2, then becomes small. What ARMA models should I try first? Explain the reasoning and the limitations.

### Prompt 2: Residual diagnostics

> I fit an ARMA model and the residual ACF still has significant spikes at lags 1 and 2. What could this mean? What should I check next?

### Prompt 3: Information criteria

> AIC chooses ARMA(2,2), but BIC chooses ARMA(1,1). How should I decide which model to report for a forecasting project?

### Prompt 4: Leakage check

> I want to evaluate time-series forecasts. Explain why random train/test splitting is usually inappropriate and how to use chronological validation instead.

When using AI, ask it to explain assumptions, not just produce code. Then test the code yourself.

## 13. Lab checklist

Before you finish, make sure you can do the following without looking at the solutions:

- [ ] explain the difference between model identification, estimation, and diagnostics;
- [ ] describe the ACF/PACF patterns for AR, MA, and ARMA models;
- [ ] implement sample autocovariance and sample ACF from scratch;
- [ ] estimate an AR model using Yule--Walker equations;
- [ ] fit ARMA models using `ARIMA(..., order=(p,0,q))`;
- [ ] compare models using AIC and BIC;
- [ ] check residuals using plots, residual ACF, and Ljung--Box tests;
- [ ] evaluate forecasts using a chronological train/test split;
- [ ] explain why residual diagnostics are essential even when AIC is small.

## End of Lab 7